### **Prerequisites for the Lab**

Before the session, ensure you have the environment set up.

1. Install [Ollama](https://ollama.com/) and download a lightweight model: `ollama run gemma4:latest` (or `llama3.1` or `mistral`).
2. Install the required Python libraries:

In [12]:
# Step 1: Install system compression tools and dependencies
!sudo apt-get update && sudo apt-get install -y zstd

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [13]:
# Step 2: Download and install the core Ollama environment binary
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [14]:
import os, subprocess, time

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

# Open a log file to catch the output so it doesn't fill the OS pipe
log_file = open('/content/ollama_server.log', 'w')

subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=log_file
)

time.sleep(5)

In [15]:
!ollama pull gemma4:latest

In [16]:
!pip install langchain langchain-ollama langchain-community

In [17]:
import urllib.request
import xml.etree.ElementTree as ET
import requests
import json

In [18]:
# --- LAB CONFIGURATION ---
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "gemma4:latest"

In [19]:
def query_agent(role_prompt, user_input, require_json=False):
    """Helper function to send prompts to the local open-source LLM."""
    payload = {
        "model": MODEL,
        "prompt": user_input,
        "system": role_prompt,
        "stream": False
    }
    if require_json:
        payload["format"] = "json"

    response = requests.post(OLLAMA_URL, json=payload)
    return response.json().get("response", "")

### TOOL: THE SYSTEM EXECUTION ENVIRONMENT

In [20]:
def arxiv_search_tool(query, max_results=2):
    """External API Tool: Fetches real research papers from ArXiv."""
    print(f"\n[System Execution] Calling ArXiv API for: '{query}'...")
    safe_query = query.replace(" ", "+")
    url = f"http://export.arxiv.org/api/query?search_query=all:{safe_query}&start=0&max_results={max_results}"

    try:
        response = urllib.request.urlopen(url)
        raw_data = response.read().decode('utf-8')
        root = ET.fromstring(raw_data)
        namespace = {'atom': 'http://www.w3.org/2005/Atom'}

        results = []
        for entry in root.findall('atom:entry', namespace):
            title = entry.find('atom:title', namespace).text.strip()
            summary = entry.find('atom:summary', namespace).text.strip().replace("\n", " ")
            results.append(f"Title: {title}\nAbstract: {summary}\n")

        return "\n".join(results)
    except Exception as e:
        return f"Error executing search: {e}"

### THE AGENTIC LIFECYCLE

In [21]:
def run_research_assistant(user_goal):
    print(f"USER GOAL: {user_goal}")

    # ---------------------------------------------------------
    # PHASE 1: PERCEPTION & PLANNING
    # The LLM acts as a reasoning engine to decide what tool to use.
    # ---------------------------------------------------------
    planning_prompt = """You are the Planning Agent. Extract the core search topic from the user's request.
    Output strictly as JSON using this schema: {"tool": "arxiv_search", "query": "extracted topic keywords"}"""

    print("[Agent State] Planning strategy and generating JSON payload...")
    plan_json_str = query_agent(planning_prompt, user_goal, require_json=True)

    try:
        plan = json.loads(plan_json_str)
        search_query = plan.get("query", "")
    except json.JSONDecodeError:
        print("Error: Agent failed to produce valid JSON.")
        return

    # ---------------------------------------------------------
    # PHASE 2: HUMAN-IN-THE-LOOP (HITL)
    # Level 2 Autonomy: The human approves the plan before execution.
    # ---------------------------------------------------------
    print(f"\nHUMAN-IN-THE-LOOP GATE")
    print(f"The Agent wants to execute the tool: '{plan.get('tool')}' with query: '{search_query}'")
    approval = input("Do you approve this execution? (y/n): ")

    if approval.lower() != 'y':
        print("Execution blocked by human guardrail. Aborting.")
        return

    # ---------------------------------------------------------
    # PHASE 3: EXECUTION -> THE SCOUT
    # The host environment executes the requested tool.
    # ---------------------------------------------------------
    raw_data = arxiv_search_tool(search_query)

    # ---------------------------------------------------------
    # PHASE 4: EVALUATION -> THE ANALYST
    # ---------------------------------------------------------
    print("\n[Agent State] The Analyst is reading raw data and checking facts...")
    analyst_prompt = f"""You are the Analyst Agent. Read the following raw research abstracts.
    Extract the key findings related to '{search_query}'. Discard any irrelevant information.
    Keep your notes concise and factual."""

    analyst_notes = query_agent(analyst_prompt, raw_data)

    # ---------------------------------------------------------
    # PHASE 5: SYNTHESIS -> THE EDITOR
    # ---------------------------------------------------------
    print("[Agent State] The Editor is formatting the final report...")
    editor_prompt = """You are the Editor Agent. Take the provided research notes and synthesize
    them into a highly professional, 3-bullet-point executive summary."""

    final_report = query_agent(editor_prompt, analyst_notes)

    print("FINAL EXECUTIVE SUMMARY")
    print(final_report)

In [22]:
# --- RUN THE Agent---
run_research_assistant("Find and summarize recent 6G wireless network research papers.")

USER GOAL: Find and summarize recent 6G wireless network research papers.
[Agent State] Planning strategy and generating JSON payload...

HUMAN-IN-THE-LOOP GATE
The Agent wants to execute the tool: 'arxiv_search' with query: '6G wireless networks research'
Do you approve this execution? (y/n): y

[System Execution] Calling ArXiv API for: '6G wireless networks research'...

[Agent State] The Analyst is reading raw data and checking facts...
[Agent State] The Editor is formatting the final report...
FINAL EXECUTIVE SUMMARY
**Executive Summary: Key Research Directions in 6G Wireless Networks**

*   **Deployment of Intelligent Reflecting Surfaces (IRSs):** IRS technology is identified as a critical enabler for 6G networks, providing novel mechanisms to enhance spectrum and energy efficiencies. Current research focuses on maximizing data rates, optimizing power spectral utilization, and integrating deep learning models for secure communication enhancement and precise terminal positioning.
*